In [1]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Folder with the emoji PNGs (change if yours is different)
emoji_dir = "data/G-emojis"

emoji_files = sorted(os.listdir(emoji_dir))
print("Number of emojis:", len(emoji_files))

# Quick sanity check: show a few filenames
emoji_files[:5]

FileNotFoundError: [Errno 2] No such file or directory: 'data/G-emojis'

In [ ]:
import numpy as np
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image

# Pretrained CNN we use as a "feature extractor" (not a classifier)
model = EfficientNetB3(weights="imagenet", include_top=False, pooling="avg")

In [ ]:
def extract_features(img_path, target_size=(384, 384)):
    """Load one PNG and return a 1D feature vector from EfficientNetB3."""
    img = image.load_img(img_path, target_size=target_size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)

    vec = model.predict(arr, verbose=0)
    return vec.flatten()

In [ ]:
features = []
valid_files = []

print("Extracting features...")
for i, fname in enumerate(emoji_files):
    if i % 500 == 0:
        print(f"  {i}/{len(emoji_files)}")

    path = os.path.join(emoji_dir, fname)
    try:
        feat = extract_features(path)
        features.append(feat)
        valid_files.append(fname)
    except Exception as e:
        # If an image is bad/corrupt, skip it
        print("Skipping", fname, "-", e)

features = np.array(features)
print("✅ Feature matrix shape:", features.shape)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

print("Finding a good K with the silhouette method...")

# PCA makes the vectors smaller + faster for clustering
pca = PCA(n_components=50, random_state=42)
features_pca = pca.fit_transform(features)
print("PCA variance explained:", round(pca.explained_variance_ratio_.sum(), 3))

k_values = list(range(2, 51))
scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(features_pca)
    scores.append(silhouette_score(features_pca, labels))

best_k = k_values[int(np.argmax(scores))]
print("Best K:", best_k, "(silhouette:", round(max(scores), 4), ")")

plt.figure(figsize=(10, 4))
plt.plot(k_values, scores)
plt.axvline(best_k, linestyle='--')
plt.xlabel("K")
plt.ylabel("Silhouette")
plt.title("Silhouette Score vs K")
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# If you ran the silhouette cell above, use that K. Otherwise fall back to 50.
n_clusters = int(globals().get("best_k", 50))
print(f"Clustering {len(features)} emojis into {n_clusters} clusters...")

kmeans = KMeans(n_clusters=n_clusters, random_state=67, n_init=10)
emoji_clusters = kmeans.fit_predict(features)

print("Emojis per cluster:")
for cid in range(n_clusters):
    count = int((emoji_clusters == cid).sum())
    print(f"  Cluster {cid}: {count}")

In [ ]:
import math
import os
from PIL import Image
import matplotlib.pyplot as plt

# Show each cluster as a grid of images
max_cols = 12
img_scale = 0.9

for cid in range(n_clusters):
    idxs = [i for i, lab in enumerate(emoji_clusters) if lab == cid]
    if not idxs:
        continue

    n = len(idxs)
    rows = math.ceil(n / max_cols)

    fig, axes = plt.subplots(rows, max_cols, figsize=(max_cols * img_scale, rows * img_scale))
    fig.suptitle(f"Cluster {cid} ({n} emojis)", y=1.02)

    if rows == 1:
        axes = axes.reshape(1, -1)

    for j, emoji_i in enumerate(idxs):
        r, c = divmod(j, max_cols)
        img_path = os.path.join(emoji_dir, valid_files[emoji_i])
        axes[r, c].imshow(Image.open(img_path))
        axes[r, c].axis("off")

    # Turn off any empty slots
    for j in range(n, rows * max_cols):
        r, c = divmod(j, max_cols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()

print("✅ Done showing clusters")

In [ ]:
# Optional: save the last figure you showed
# fig.savefig('results.png', dpi=fig.dpi)

In [ ]:
import os

def find_png_folder(base="."):
    """Search for a folder that contains lots of PNG files."""
    hits = []
    for root, _, files in os.walk(base):
        pngs = [f for f in files if f.lower().endswith(".png")]
        if len(pngs) >= 50:
            hits.append((root, len(pngs)))
    hits.sort(key=lambda x: x[1], reverse=True)
    return hits

find_png_folder(".")[:5]

In [ ]:
import os
import shutil
import json
import random
from datetime import datetime

# Pick a random subset of emojis so we can do a smaller human study
N = 60
SEED = 42

random.seed(SEED)

out_dir = "subset_60"
os.makedirs(out_dir, exist_ok=True)

random_picks = random.sample(range(len(valid_files)), N)
idx_to_original = {}

for new_i, old_i in enumerate(random_picks):
    src = os.path.join(emoji_dir, valid_files[old_i])
    dst_name = f"emoji_{new_i:03d}.png"
    dst = os.path.join(out_dir, dst_name)
    shutil.copy2(src, dst)
    idx_to_original[dst_name] = valid_files[old_i]

with open("subset_mapping.json", "w") as f:
    json.dump(idx_to_original, f, indent=2)

print(f"✅ Saved {N} emojis to {out_dir}/ and wrote subset_mapping.json")

In [ ]:
import json

# Make a simple AI assignment file for the 60 emoji subset
# (cluster label is whatever KMeans predicted for that emoji)
ai_assignments = {}
for new_i, old_i in enumerate(random_picks):
    fname = f"emoji_{new_i:03d}.png"
    ai_assignments[fname] = int(emoji_clusters[old_i])

with open("ai_assignments.json", "w") as f:
    json.dump(ai_assignments, f, indent=2)

print("✅ Wrote ai_assignments.json")

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Re-extract features from the subset folder (what the website uses)
subset_dir = "subset_60"
subset_files = sorted([f for f in os.listdir(subset_dir) if f.endswith('.png')])

subset_features = []
for f in subset_files:
    subset_features.append(extract_features(os.path.join(subset_dir, f)))
subset_features = np.array(subset_features)
print("✅ Subset features:", subset_features.shape)

In [ ]:
from sklearn.cluster import KMeans

# Pick a k that gives a reasonable number of groups for comparison
best_k = 5

km_subset = KMeans(n_clusters=best_k, random_state=67, n_init=10)
subset_labels = km_subset.fit_predict(subset_features)

print("Cluster counts:")
for cid in range(best_k):
    print(cid, int((subset_labels == cid).sum()))

In [ ]:
import math
from PIL import Image
import matplotlib.pyplot as plt

max_cols = 10
img_scale = 1.0

for cid in range(best_k):
    idxs = [i for i, lab in enumerate(subset_labels) if lab == cid]
    if not idxs:
        continue

    n = len(idxs)
    rows = math.ceil(n / max_cols)
    fig, axes = plt.subplots(rows, max_cols, figsize=(max_cols * img_scale, rows * img_scale))
    fig.suptitle(f"Subset cluster {cid} ({n})", y=1.02)

    if rows == 1:
        axes = axes.reshape(1, -1)

    for j, i in enumerate(idxs):
        r, c = divmod(j, max_cols)
        img = Image.open(os.path.join("subset_60", subset_files[i]))
        axes[r, c].imshow(img)
        axes[r, c].axis("off")

    for j in range(n, rows * max_cols):
        r, c = divmod(j, max_cols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Old human comparison code was removed.
# The JSON-based comparison is in the next big cell.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "emoji", "--quiet"], check=False)

In [ ]:
import json
import emoji

with open("ai_assignments.json") as f:
    ai_data = json.load(f)

with open("subset_mapping.json") as f:
    idx_to_original = json.load(f)

def filename_to_emoji_name(fname):
    """Try to turn an emoji_###.png into a readable emoji name (best effort)."""
    # This notebook stores subset filenames, so we just return the filename for now.
    # If you have unicode filenames, you could decode them here.
    return fname

for i in range(60):
    fname = f"emoji_{i:03d}.png"
    orig = idx_to_original.get(fname, "NOT FOUND")
    print(f"{fname} -> {orig}")

In [ ]:
# (Optional) extra debug prints were here before.

In [ ]:
import json
import numpy as np
from itertools import combinations
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---- Load AI clusters (our labels) ----
with open("ai_assignments.json") as f:
    ai_assignments = json.load(f)

# ---- Load human card-sort JSON files ----
# Put your participant JSON exports in ./participant_json/
human_dir = "participant_json"
human_files = [f for f in os.listdir(human_dir) if f.endswith('.json')]

print("Found human files:", human_files)

# Helper: read one participant file and return {emoji_filename: cluster_id}
def load_human_clusters(path):
    data = json.load(open(path))

    # This assumes your export format is like: {"clusters": {"name": ["emoji_000.png", ...], ...}}
    # If your JSON has a different shape, adjust this function.
    clusters = data.get("clusters", data)

    mapping = {}
    for cid, (cluster_name, items) in enumerate(clusters.items()):
        for fname in items:
            mapping[fname] = cid
    return mapping

# Convert AI labels to aligned arrays
all_items = sorted(ai_assignments.keys())
ai_labels = np.array([ai_assignments[k] for k in all_items])

print("
Participant vs AI clustering")
print("-"*70)
print(f"{'participant':<12} {'ARI':>6} {'NMI':>6}  file")

for hf in human_files:
    path = os.path.join(human_dir, hf)
    human_map = load_human_clusters(path)

    # Only compare items that exist in both
    common = [k for k in all_items if k in human_map]
    if not common:
        continue

    h_labels = np.array([human_map[k] for k in common])
    a_labels = np.array([ai_assignments[k] for k in common])

    ari = adjusted_rand_score(h_labels, a_labels)
    nmi = normalized_mutual_info_score(h_labels, a_labels)

    name = os.path.splitext(hf)[0][:12]
    print(f"{name:<12} {ari:6.3f} {nmi:6.3f}  {hf}")

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers", "--quiet"], check=False)

In [ ]:
import json

with open("ai_assignments.json") as f:
    ai_raw = json.load(f)

items = sorted(ai_raw.keys())
print("Items:", len(items))

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading text model...")
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Turn filenames into simple text (you can replace this with real emoji names)
texts = items
text_features = text_model.encode(texts, show_progress_bar=True)

k_values = list(range(2, 16))
scores = []
for k in k_values:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(text_features)
    scores.append(silhouette_score(text_features, labels))

best_k_text = k_values[int(np.argmax(scores))]
print("Best k (text):", best_k_text)

plt.figure(figsize=(8, 4))
plt.plot(k_values, scores)
plt.axvline(best_k_text, linestyle='--')
plt.xlabel('K')
plt.ylabel('Silhouette')
plt.title('Silhouette vs K (text)')
plt.show()

In [ ]:
# If you want REAL emoji names for the full dataset, you can build that list here.
# The original notebook tried to decode unicode filenames.
# Keeping this cell as a placeholder so the notebook stays simple.

In [ ]:
# Placeholder for encoding the full dataset of emoji names.
# If you add `full_emoji_names`, you can do:
# full_text_features = text_model.encode(full_emoji_names, show_progress_bar=True)

In [ ]:
# Placeholder for clustering the full text feature matrix.
# Example:
# km_full = KMeans(n_clusters=best_k_full, random_state=42, n_init=10)
# full_text_labels = km_full.fit_predict(full_text_features)

In [ ]:
# Optional: inspect what a small K (like 2) actually groups together.